# 02 — Instance, Class, and Static Methods + `__repr__`/`__str__`

All three method kinds live inside a `class` block, but they behave very differently. Knowing which to use is the difference between code that reads naturally and code that has spurious `self`s everywhere.

## Instance methods — the default

First parameter is `self` — the instance. Use when the method needs the instance's state.

In [1]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        # Uses self.radius — must be an instance method.
        return 3.14159 * self.radius ** 2

c = Circle(5)
print(c.area())

78.53975


## `@classmethod` — receives the class, not the instance

First parameter is `cls`. The method *knows what class it's on* but not any particular instance. The big use case is **alternative constructors** — extra ways to build an instance from different inputs.

This pattern is everywhere — `dict.fromkeys`, `datetime.fromisoformat`, Pydantic's `model_validate`, FastAPI dependency factories.

In [2]:
class Circle:
    PI = 3.14159

    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return Circle.PI * self.radius ** 2

    @classmethod
    def from_diameter(cls, d):
        # `cls` is `Circle` here. Returning `cls(...)` lets subclasses get instances of
        # themselves rather than always Circle — small but important.
        return cls(d / 2)

    @classmethod
    def unit(cls):
        return cls(1)

c1 = Circle(5)
c2 = Circle.from_diameter(10)
c3 = Circle.unit()

print(c1.area(), c2.area(), c3.area())

78.53975 78.53975 3.14159


## `@staticmethod` — receives nothing automatic

No `self`, no `cls`. It's a plain function that happens to live inside a class because it's *logically grouped* with that class — but it doesn't need any of its state.

If you find yourself reaching for `@staticmethod`, ask: does this really belong on the class, or should it just be a module-level function? Often the latter is cleaner.

In [3]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @staticmethod
    def c_to_f(c):
        # Doesn't use self or cls — just a related utility.
        return c * 9 / 5 + 32

    def fahrenheit(self):
        return Temperature.c_to_f(self.celsius)

t = Temperature(100)
print(t.fahrenheit())
print(Temperature.c_to_f(0))     # callable without an instance

212.0
32.0


## Decision flow

```
Does it use self.something?
├── Yes  → instance method
└── No
    ├── Does it create or return an instance of the class?
    │   └── Yes → @classmethod   (alternative constructor)
    └── No  → @staticmethod      (or: just make it a module-level function)
```

## `__repr__` and `__str__` — make your objects printable

Without these, `print(obj)` shows `<Class object at 0x...>` — useless. Two dunders solve this:

- **`__repr__`** — the *developer*-facing representation. Should be unambiguous, ideally `eval(repr(x)) == x` (when feasible). Used by the REPL, by lists/dicts of objects, by debuggers.
- **`__str__`** — the *user*-facing representation. Used by `print()` and `f"{obj}"`. If you don't define it, Python falls back to `__repr__`.

**Always define `__repr__`. Define `__str__` only if the user-facing form differs from the developer form.**

In [4]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x!r}, y={self.y!r})"

p = Point(3, 4)
print(repr(p))     # Point(x=3, y=4)
print(p)           # uses __str__, which falls back to __repr__
print([p, p])      # lists call __repr__ on each element

Point(x=3, y=4)
Point(x=3, y=4)
[Point(x=3, y=4), Point(x=3, y=4)]


In [5]:
class Money:
    def __init__(self, amount, currency):
        self.amount = amount
        self.currency = currency

    def __repr__(self):
        return f"Money(amount={self.amount!r}, currency={self.currency!r})"

    def __str__(self):
        return f"{self.amount:.2f} {self.currency}"

m = Money(1234.5, "USD")
print(repr(m))   # debug-friendly
print(m)         # user-friendly: 1234.50 USD
print(f"price = {m}")

Money(amount=1234.5, currency='USD')
1234.50 USD
price = 1234.50 USD


### `!r` in f-strings

`f"{x!r}"` calls `repr(x)` instead of `str(x)`. Use it in `__repr__` so that string fields show with their quotes:

In [6]:
name = "Alice"
print(f"{name}")     # Alice
print(f"{name!r}")   # 'Alice'

# In a __repr__, the !r form lets a reader tell str(3) from int 3 from a Point named "3".
# It's a 1-character habit that pays off forever.

Alice
'Alice'


## Putting it together — a small but real-looking class

In [7]:
class Temperature:
    ABSOLUTE_ZERO_C = -273.15   # class attribute (immutable — fine)

    def __init__(self, celsius):
        if celsius < self.ABSOLUTE_ZERO_C:
            raise ValueError(f"below absolute zero: {celsius}")
        self.celsius = celsius

    # ---- alternative constructors ----
    @classmethod
    def from_fahrenheit(cls, f):
        return cls((f - 32) * 5 / 9)

    @classmethod
    def from_kelvin(cls, k):
        return cls(k + cls.ABSOLUTE_ZERO_C)

    # ---- utility (no self / cls) ----
    @staticmethod
    def is_freezing(c):
        return c <= 0

    # ---- instance behavior ----
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    def kelvin(self):
        return self.celsius - self.ABSOLUTE_ZERO_C

    # ---- printability ----
    def __repr__(self):
        return f"Temperature(celsius={self.celsius!r})"

    def __str__(self):
        return f"{self.celsius:.1f} °C"


boiling = Temperature(100)
room    = Temperature.from_fahrenheit(72)
absolute_ish = Temperature.from_kelvin(0)

for t in [boiling, room, absolute_ish]:
    print(f"{t!r}  -> {t}  (F={t.fahrenheit():.1f}, K={t.kelvin():.1f}, freezing? {Temperature.is_freezing(t.celsius)})")

Temperature(celsius=100)  -> 100.0 °C  (F=212.0, K=373.1, freezing? False)
Temperature(celsius=22.22222222222222)  -> 22.2 °C  (F=72.0, K=295.4, freezing? False)
Temperature(celsius=-273.15)  -> -273.1 °C  (F=-459.7, K=0.0, freezing? True)


## Mini-exercise

1. Add an alternative constructor `Money.from_str("1234.50 USD")` that parses the string back.
2. To `Point` (above), add `Point.origin()` (classmethod) and `Point.midpoint(a, b)` — should the latter be a `@classmethod` or `@staticmethod`? Justify.
3. Take an existing class without `__repr__` (like `Circle` from earlier in this notebook). Print it, print a list of two of them. Then add a `__repr__` and observe the difference.

In [8]:
# your solutions here


**Takeaways**

- Instance method (`self`) for behavior that uses instance state.
- `@classmethod` (`cls`) for alternative constructors — `cls(...)` is sub-class-friendly.
- `@staticmethod` for utilities grouped with the class; consider a module function instead.
- **Always** define `__repr__`. Add `__str__` only when the user form differs.
- `f"{x!r}"` uses `repr(x)`; use it inside `__repr__` for clarity.

Next: [03_bank_account/](03_bank_account/) — a real `.py` class.